# VLM hallucination probe: Colab runner

Entry point for the whole pipeline. Every cell just calls a stage script in `src/scripts/`; all logic lives in the `lib` package.

**Runtime:** Runtime → Change runtime type → **T4 GPU**.

All outputs go to Google Drive, so after a disconnect re-run the setup cells and then the stage you were on. Every long stage resumes where it stopped.

## 0. Setup

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# If the repo is private, use https://<GITHUB_TOKEN>@github.com/JeremyQuek/4001Project.git
REPO = 'https://github.com/JeremyQuek/4001Project.git'
BRANCH = 'jeremy'

import os
if not os.path.exists('/content/4001Project'):
    !git clone -b {BRANCH} {REPO} /content/4001Project
else:
    !git -C /content/4001Project pull
%cd /content/4001Project/src
!pip install -q -e .
import nltk
nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True)

In [ ]:
# Where data and results live on Drive. RUN names the experiment folder.
DRIVE = '/content/drive/MyDrive/vlm_halluc'
RUN = 'default'
COMMON = f'--set paths.data_root={DRIVE}/data --set paths.out_root={DRIVE}/runs --set run_name={RUN}'
# Add overrides here if the pilot says so, e.g.
# COMMON += ' --set model.dtype=float32'
# COMMON += ' --set model.max_pixels=200704'
OUT = f'{DRIVE}/runs/{RUN}'
print(COMMON)

## 1. Data: COCO annotations, POPE exclusion, splits, image download
CPU only. About 10 minutes the first time.

In [ ]:
!python scripts/01_prepare_data.py {COMMON}

## 2a. Pilot (100 images): check before the full run
Look at the printout:
- **Near-empty captions** → fp16 is broken: add `--set model.dtype=float32` to `COMMON` and rerun.
- **Estimated hallucinated mentions < 500** → add `--set model.max_pixels=200704` or `--set data.n_images=8000`.

To rerun the pilot after changing settings, delete `OUT/pilot/` first.

In [ ]:
!python scripts/02_generate_captions.py {COMMON} --pilot

## 2b. Captions for all images (longest GPU job, ~1–3 h)
Resumable: if Colab disconnects, run setup again and rerun this cell.

In [ ]:
!python scripts/02_generate_captions.py {COMMON}

## 3. CHAIR labels + label-noise audit sheet

In [ ]:
!python scripts/03_label.py {COMMON}
!python scripts/label_audit.py export {COMMON}
# Two people fill annotator_1 / annotator_2 in OUT/label_audit.csv, then:
# !python scripts/label_audit.py score {COMMON}

## 4. Activations (teacher-forced pass)
To also record attention-to-image for that baseline, add `--set activations.attention=true`. It is slower and uses more memory.

In [ ]:
!python scripts/04_extract_activations.py {COMMON}

## 5. Train probes (every layer × position)
Add `--skip-mlp` for a fast logistic-regression-only run.

In [ ]:
!python scripts/05_train_probes.py {COMMON}

## 6. Baselines
`--owlvit` and `--selfcheck` need the GPU; drop them if short on time.

In [ ]:
!python scripts/06_baselines.py {COMMON} --owlvit --selfcheck

## 7. Part 3 base captions (+ optional during-generation correction)
Without `--guided` this only makes the no-correction captions that step 8 needs. `--guided` adds the candidate/skip policies, which take several hours.

In [ ]:
!python scripts/07_guided_generation.py {COMMON}
# !python scripts/07_guided_generation.py {COMMON} --guided

## 8. After-generation correction (feedback loop + controls)

In [ ]:
!python scripts/08_feedback_loop.py {COMMON}

## 9. Tables and figures

In [ ]:
!python scripts/09_evaluate.py {COMMON}
from IPython.display import Image, display
import glob
for f in sorted(glob.glob(f'{OUT}/figures/*.png')):
    print(f); display(Image(f, width=600))